# 05 — Final Qwen test + frozen validation calibration check

Run once after reviewing notebook 01 and freezing model choices. Add Input → Notebook Output → the saved output of 01. Enable GPU and Internet, then Run All. It evaluates the same pinned Qwen checkpoint on test; temperature is read from validation and never fitted to test.

In [ ]:
import json
from pathlib import Path
candidates = [p for p in Path('/kaggle/input').rglob('validation_predictions.jsonl') if (p.parent / 'scored/scorecard.json').is_file()]
if len(candidates) != 1:
    raise RuntimeError('Attach exactly the saved output of notebook 01 via Add Input → Notebook Output.')
validation_raw = candidates[0]
frozen_scorecard_path = validation_raw.parent / 'scored/scorecard.json'
frozen_scorecard = json.loads(frozen_scorecard_path.read_text())


# SatQuery Qwen3-VL — evaluation only

Use this notebook after training. It downloads the exact public adapter and immutable base
revision, reconstructs a leakage-safe BigEarthNet.txt validation/test subset, evaluates every
supported task, compares the pinned base against base+LoRA on the exact same inputs, and exports
raw predictions plus real generation scores. It performs **no training** and provisions **no
paid endpoint**. Run on a free Kaggle/Colab GPU; stop the GPU after the final PASS cell.

## 0. Install the pinned evaluation environment

In [ ]:
import subprocess
import sys

PACKAGES = [
    "transformers==4.57.1",
    "accelerate==1.7.0",
    "peft==0.17.1",
    "bitsandbytes==0.47.0",
    "huggingface_hub==0.36.2",
    "qwen-vl-utils==0.0.14",
    "duckdb==1.3.2",
    "pandas>=2.2,<3",
    "pyarrow>=18,<23",
    "lmdb>=1.5,<2",
    "safetensors>=0.5,<1",
    "tqdm>=4.66,<5",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PACKAGES])
print("Installed. Restart the runtime once only if a later import reports a cached old package.")

## 1. Configuration

`validation` is the repeatable development benchmark. Run `test` once for the final report.
The default 200 examples is a bounded evidence run that fits a free T4 session. Each example is
evaluated twice without loading a second copy of the model: PEFT temporarily disables the adapter
for the base pass.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import os


@dataclass(frozen=True)
class Config:
    adapter_repo: str = "aanandmodi/satquery-qwen3vl-bigearthnet-txt-lora"
    adapter_revision: str = "ed12e59e0def9468bdf4a226789fc1b77c7900e7"
    base_model: str = "Qwen/Qwen3-VL-2B-Instruct"
    base_revision: str = "89644892e4d85e24eaac8bacfd4f463576704203"
    text_repo: str = "BIFOLD-BigEarthNetv2-0/BigEarthNet.txt"
    text_revision: str = "72d865f2146f0a85b720f7f3ca1cdbaeafc3d316"
    image_repo: str = "hackelle/BigEarthNetV2-Lithuania-Summer-LMDB"
    image_revision: str = "7a83ae701109ec232d40665b9677fc46310a1a8b"
    split: str = "test"  # validation or test
    rows_per_type: int = 50
    seed: int = 42
    image_size: int = 448
    max_new_tokens: int = 128
    upload_results: bool = False


CFG = Config()
assert CFG.split in {"validation", "test"}
ROOT = Path("/kaggle/working/satquery-eval" if Path("/kaggle/working").exists() else "/content/satquery-eval")
DATA_DIR, OUTPUT_DIR = ROOT / "data", ROOT / "outputs"
for path in (DATA_DIR, OUTPUT_DIR):
    path.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(ROOT / "hf-cache"))
print(CFG)

## 2. GPU and artifact integrity gates

In [ ]:
import json
import random
import numpy as np
import torch
from huggingface_hub import HfApi, hf_hub_download, snapshot_download

random.seed(CFG.seed)
np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle/Colab GPU before continuing.")

api = HfApi()
adapter_info = api.model_info(CFG.adapter_repo, revision=CFG.adapter_revision)
assert adapter_info.sha == CFG.adapter_revision
base_revision_file = Path(hf_hub_download(
    CFG.adapter_repo, "base_revision.txt", revision=CFG.adapter_revision
))
assert base_revision_file.read_text(encoding="utf-8").strip() == CFG.base_revision
print({"gpu": torch.cuda.get_device_name(0), "adapter_sha": adapter_info.sha, "base_sha": CFG.base_revision})

## 3. Download and join text records to matching imagery

The user-supplied dataset is a text/QA table, not the image pixels. The 2.48 GB LMDB mirror below
supplies matching Sentinel-2 imagery by patch ID. The SQL query preserves the publisher split and
applies a deterministic per-task cap without loading all 9.55 million text rows into RAM.

In [ ]:
text_path = Path(hf_hub_download(
    CFG.text_repo,
    "BigEarthNet.txt.parquet",
    repo_type="dataset",
    revision=CFG.text_revision,
    local_dir=DATA_DIR / "text",
))
image_root = Path(snapshot_download(
    CFG.image_repo,
    repo_type="dataset",
    revision=CFG.image_revision,
    local_dir=DATA_DIR / "images",
    allow_patterns=["BENv2_lithuania_summer.lmdb/*", "metadata_lithuania_summer.parquet"],
))
lmdb_dir = image_root / "BENv2_lithuania_summer.lmdb"
metadata_path = image_root / "metadata_lithuania_summer.parquet"
required = [text_path, metadata_path, lmdb_dir / "data.mdb", lmdb_dir / "lock.mdb"]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Incomplete dataset download: {missing}")

In [ ]:
import duckdb
import pandas as pd


def sql_path(path: Path) -> str:
    return str(path.resolve()).replace("'", "''")


prepared_path = DATA_DIR / f"{CFG.split}-evaluation.parquet"
duckdb.sql(f"""
COPY (
  WITH matched AS (
    SELECT t.ID, t.patch_id, t.input, t.output, t.type, t.category, t.split
    FROM read_parquet('{sql_path(text_path)}') t
    SEMI JOIN read_parquet('{sql_path(metadata_path)}') m USING (patch_id)
    WHERE t.split = '{CFG.split}'
      AND t.type IN ('binary', 'mcq', 'captioning', 'bounding box')
      AND lower(t.category) NOT IN ('country','season','climate zone','climate_zone','cliamte zone')
  ), ranked AS (
    SELECT *, row_number() OVER (
      PARTITION BY type ORDER BY hash(CAST(ID AS VARCHAR) || '{CFG.seed}')
    ) AS rank
    FROM matched
  )
  SELECT * EXCLUDE (rank) FROM ranked WHERE rank <= {CFG.rows_per_type}
) TO '{sql_path(prepared_path)}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")
frame = pd.read_parquet(prepared_path)
assert frame.ID.is_unique and not frame[["patch_id", "input", "output", "type"]].isna().any().any()
assert set(frame.type) == {"binary", "mcq", "captioning", "bounding box"}
print(frame.groupby("type").size())

## 4. Safe LMDB image reader and coordinate conversion

In [ ]:
import lmdb
import re
from PIL import Image
from safetensors.numpy import load as load_safetensors_bytes

S2_BANDS = ["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B09", "B11", "B12"]
NUMBER = r"[-+]?\d*\.?\d+"


class ImageStore:
    def __init__(self, path: Path):
        self.path = path
        self._env = None

    @property
    def env(self):
        if self._env is None:
            self._env = lmdb.open(str(self.path), readonly=True, lock=False, readahead=True, meminit=False)
        return self._env

    def rgb(self, patch_id: str) -> Image.Image:
        with self.env.begin(write=False, buffers=True) as transaction:
            raw = transaction.get(str(patch_id).encode("utf-8"))
        if raw is None:
            raise KeyError(f"LMDB image missing for {patch_id}")
        bands = load_safetensors_bytes(bytes(raw))
        if not set(S2_BANDS).issubset(bands):
            raise ValueError(f"S2 bands missing for {patch_id}")
        channels = []
        for name in ("B04", "B03", "B02"):
            band = np.asarray(bands[name], dtype=np.float32)
            low, high = np.nanpercentile(band, (2, 98))
            channels.append((np.clip((band - low) / max(high - low, 1e-6), 0, 1) * 255).astype(np.uint8))
        image = Image.fromarray(np.stack(channels, axis=-1))
        return image.resize((CFG.image_size, CFG.image_size), Image.Resampling.BILINEAR)


def qwen_coordinate(value: float) -> int:
    if not 0 <= value <= 1:
        raise ValueError(f"Coordinate outside [0,1]: {value}")
    return int(round(1000 * value))


def convert_question(text: str) -> str:
    pattern = re.compile(rf"<point>\s*\(({NUMBER})\s*,\s*({NUMBER})\)\s*</point>")
    return pattern.sub(lambda m: f"<point>({qwen_coordinate(float(m[1]))},{qwen_coordinate(float(m[2]))})</point>", text)


def reference_answer(row: pd.Series) -> str:
    if row["type"] != "bounding box":
        return str(row["output"]).strip()
    values = [float(item) for item in re.findall(NUMBER, str(row["output"]))]
    if len(values) != 4 or not (0 <= values[0] <= values[2] <= 1 and 0 <= values[1] <= values[3] <= 1):
        raise ValueError(f"Invalid reference box: {row['output']}")
    box = [qwen_coordinate(item) for item in values]
    return f"<box>({box[0]},{box[1]}),({box[2]},{box[3]})</box>"


images = ImageStore(lmdb_dir)
display(images.rgb(str(frame.iloc[0].patch_id)))
print(convert_question(str(frame.iloc[0].input)), reference_answer(frame.iloc[0]))

## 5. Load the published adapter in 4-bit and run a smoke test

In [ ]:
from peft import PeftModel
from qwen_vl_utils import process_vision_info
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration

quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
processor = AutoProcessor.from_pretrained(CFG.adapter_repo, revision=CFG.adapter_revision, trust_remote_code=False)
base = Qwen3VLForConditionalGeneration.from_pretrained(
    CFG.base_model,
    revision=CFG.base_revision,
    trust_remote_code=False,
    quantization_config=quantization,
    device_map="auto",
)
model = PeftModel.from_pretrained(base, CFG.adapter_repo, revision=CFG.adapter_revision, is_trainable=False)
model.eval()


def prepared_inputs(image: Image.Image, question: str):
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image}, {"type": "text", "text": question},
    ]}]
    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    batch = processor(text=[prompt], images=image_inputs, videos=video_inputs, return_tensors="pt")
    batch = {key: value.to(model.device) for key, value in batch.items()}
    return batch


@torch.inference_mode()
def generate_scored(batch) -> tuple[str, float, float]:
    output = model.generate(
        **batch,
        max_new_tokens=CFG.max_new_tokens,
        do_sample=False,
        use_cache=True,
        return_dict_in_generate=True,
        output_scores=True,
    )
    generated = output.sequences[:, batch["input_ids"].shape[1]:]
    text = processor.batch_decode(generated, skip_special_tokens=True)[0].strip()
    transition = model.compute_transition_scores(
        output.sequences, output.scores, normalize_logits=True
    )[0]
    # This is a real generation statistic, not a calibrated correctness probability.
    mean_log_probability = float(transition.mean().item()) if transition.numel() else -20.0
    sequence_confidence = float(np.clip(np.exp(mean_log_probability), 1e-6, 1 - 1e-6))
    sequence_logit = float(np.log(sequence_confidence / (1 - sequence_confidence)))
    return text, sequence_confidence, sequence_logit


def predict_base_and_lora(image: Image.Image, question: str) -> dict[str, float | str]:
    batch = prepared_inputs(image, question)
    lora_pred, lora_confidence, lora_logit = generate_scored(batch)
    with model.disable_adapter():
        base_pred, base_confidence, base_logit = generate_scored(batch)
    return {
        "base_pred": base_pred,
        "base_sequence_confidence": base_confidence,
        "base_sequence_logit": base_logit,
        "lora_pred": lora_pred,
        "lora_sequence_confidence": lora_confidence,
        "lora_sequence_logit": lora_logit,
    }


smoke = frame.iloc[0]
print({
    "question": str(smoke.input),
    "reference": reference_answer(smoke),
    **predict_base_and_lora(
        images.rgb(str(smoke.patch_id)), convert_question(str(smoke.input))
    ),
})

## 6. Full bounded evaluation

Exact match is used for binary/MCQ, IoU for grounding, and token F1 as a caption smoke metric.
These are task metrics—not a fabricated confidence score.

In [ ]:
import string
from collections import Counter
from tqdm.auto import tqdm


def normalize(text: str) -> str:
    table = str.maketrans("", "", string.punctuation.replace("<", "").replace(">", ""))
    return " ".join(str(text).lower().translate(table).split())


def parse_box(text: str):
    values = [int(round(float(item))) for item in re.findall(NUMBER, str(text))]
    if len(values) < 4:
        return None
    x1, y1, x2, y2 = values[:4]
    return (x1, y1, x2, y2) if 0 <= x1 <= x2 <= 1000 and 0 <= y1 <= y2 <= 1000 else None


def iou(left, right) -> float:
    ix1, iy1, ix2, iy2 = max(left[0], right[0]), max(left[1], right[1]), min(left[2], right[2]), min(left[3], right[3])
    intersection = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area_left = max(0, left[2] - left[0]) * max(0, left[3] - left[1])
    area_right = max(0, right[2] - right[0]) * max(0, right[3] - right[1])
    return intersection / max(area_left + area_right - intersection, 1)


def token_f1(prediction: str, reference: str) -> float:
    pred, ref = normalize(prediction).split(), normalize(reference).split()
    common = sum((Counter(pred) & Counter(ref)).values())
    if not pred or not ref:
        return float(pred == ref)
    return 0.0 if not common else 2 * (common / len(pred)) * (common / len(ref)) / ((common / len(pred)) + (common / len(ref)))


def score_prediction(prediction: str, reference: str, task_type: str):
    exact, box_score, caption_score = int(normalize(prediction) == normalize(reference)), None, None
    if task_type == "bounding box":
        predicted_box, reference_box = parse_box(prediction), parse_box(reference)
        box_score = iou(predicted_box, reference_box) if predicted_box and reference_box else 0.0
        exact = int(box_score >= 0.5)
    elif task_type == "captioning":
        caption_score = token_f1(prediction, reference)
    return exact, box_score, caption_score


records = []
for _, row in tqdm(frame.iterrows(), total=len(frame)):
    question = convert_question(str(row.input).strip())
    reference = reference_answer(row)
    compared = predict_base_and_lora(images.rgb(str(row.patch_id)), question)
    base_correct, base_iou, base_caption_f1 = score_prediction(
        str(compared["base_pred"]), reference, str(row.type)
    )
    lora_correct, lora_iou, lora_caption_f1 = score_prediction(
        str(compared["lora_pred"]), reference, str(row.type)
    )
    records.append({
        "id": int(row.ID), "patch_id": str(row.patch_id), "type": str(row.type),
        "question": question, "reference": reference,
        **compared,
        "base_correct": base_correct,
        "base_iou": base_iou,
        "base_caption_token_f1": base_caption_f1,
        "lora_correct": lora_correct,
        "lora_iou": lora_iou,
        "lora_caption_token_f1": lora_caption_f1,
    })

predictions = pd.DataFrame(records)
predictions.to_json(OUTPUT_DIR / f"{CFG.split}_predictions.jsonl", orient="records", lines=True)
vqa = predictions.type.isin(["binary", "mcq"])
grounding = predictions.type == "bounding box"
captioning = predictions.type == "captioning"
summary = {
    "adapter_revision": CFG.adapter_revision,
    "base_revision": CFG.base_revision,
    "dataset_revision": CFG.text_revision,
    "image_dataset_revision": CFG.image_revision,
    "split": CFG.split,
    "rows": len(predictions),
    "base": {
        "vqa_exact_match": float(predictions.loc[vqa, "base_correct"].mean()),
        "grounding_mean_iou": float(predictions.loc[grounding, "base_iou"].mean()),
        "grounding_accuracy_iou_0_5": float(predictions.loc[grounding, "base_correct"].mean()),
        "caption_token_f1": float(predictions.loc[captioning, "base_caption_token_f1"].mean()),
    },
    "lora": {
        "vqa_exact_match": float(predictions.loc[vqa, "lora_correct"].mean()),
        "grounding_mean_iou": float(predictions.loc[grounding, "lora_iou"].mean()),
        "grounding_accuracy_iou_0_5": float(predictions.loc[grounding, "lora_correct"].mean()),
        "caption_token_f1": float(predictions.loc[captioning, "lora_caption_token_f1"].mean()),
    },
    "confidence_semantics": (
        "sequence_confidence is an observed generation statistic, not a correctness probability. "
        "Use scripts/score-real-evaluation.py for held-out temperature scaling and bootstrap CIs."
    ),
}
(OUTPUT_DIR / f"{CFG.split}_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))

## 7. Optional result upload and safe stop

Uploading small metric files to the existing Hub repository is free. It is disabled by default
because the model itself is already released. If enabled, keep `HF_TOKEN` in Kaggle/Colab Secrets.

In [ ]:
if CFG.upload_results:
    token = os.environ.get("HF_TOKEN", "").strip()
    if not token:
        raise RuntimeError("Set HF_TOKEN as a notebook secret; never paste it into this cell.")
    HfApi(token=token).upload_folder(
        repo_id=CFG.adapter_repo,
        repo_type="model",
        folder_path=OUTPUT_DIR,
        path_in_repo=f"evaluation/{CFG.split}",
        commit_message=f"Add pinned {CFG.split} evaluation",
    )

gate = {
    "adapter_revision_verified": adapter_info.sha == CFG.adapter_revision,
    "predictions_written": (OUTPUT_DIR / f"{CFG.split}_predictions.jsonl").is_file(),
    "summary_written": (OUTPUT_DIR / f"{CFG.split}_summary.json").is_file(),
    "paid_endpoint_created": False,
}
print(json.dumps(gate, indent=2))
assert all(value for key, value in gate.items() if key != "paid_endpoint_created")
assert gate["paid_endpoint_created"] is False
print("PASS — evaluation complete. Download outputs, save the notebook version, and stop the GPU.")

In [ ]:
import json, runpy, sys
from pathlib import Path
scorer_root = Path('/kaggle/working/satquery-scorer')
scorer_files = {'ml/satquery_ml/__init__.py': '', 'ml/satquery_ml/evaluation.py': 'from __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Sequence\n\nimport numpy as np\n\n\ndef normalize_answer(value: str) -> str:\n    return " ".join(\n        "".join(\n            character.lower() if character.isalnum() else " " for character in value\n        ).split()\n    )\n\n\ndef exact_match(predictions: Sequence[str], references: Sequence[str]) -> float:\n    if len(predictions) != len(references) or not predictions:\n        raise ValueError(\n            "predictions and references must be non-empty and equally sized"\n        )\n    return float(\n        np.mean(\n            [\n                normalize_answer(prediction) == normalize_answer(reference)\n                for prediction, reference in zip(predictions, references, strict=True)\n            ]\n        )\n    )\n\n\ndef box_iou(box_a: Sequence[float], box_b: Sequence[float]) -> float:\n    if len(box_a) != 4 or len(box_b) != 4:\n        raise ValueError("boxes require four coordinates")\n    ax1, ay1, ax2, ay2 = box_a\n    bx1, by1, bx2, by2 = box_b\n    intersection = max(0.0, min(ax2, bx2) - max(ax1, bx1)) * max(\n        0.0, min(ay2, by2) - max(ay1, by1)\n    )\n    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)\n    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)\n    union = area_a + area_b - intersection\n    return intersection / union if union else 0.0\n\n\ndef grounding_metrics(\n    predictions: Sequence[Sequence[float]], references: Sequence[Sequence[float]]\n) -> dict[str, float]:\n    if len(predictions) != len(references) or not predictions:\n        raise ValueError("box collections must be non-empty and equally sized")\n    ious = np.array(\n        [box_iou(a, b) for a, b in zip(predictions, references, strict=True)]\n    )\n    return {\n        "mean_iou": float(ious.mean()),\n        "acc_at_0_5": float((ious >= 0.5).mean()),\n        "acc_at_0_7": float((ious >= 0.7).mean()),\n    }\n\n\ndef expected_calibration_error(\n    confidences: Sequence[float], correct: Sequence[bool], *, bins: int = 15\n) -> float:\n    confidence = np.asarray(confidences, dtype=float)\n    labels = np.asarray(correct, dtype=float)\n    if confidence.size == 0 or confidence.size != labels.size:\n        raise ValueError(\n            "confidence and correctness arrays must be equally sized and non-empty"\n        )\n    if np.any((confidence < 0) | (confidence > 1)):\n        raise ValueError("confidence values must be in 0..1")\n    edges = np.linspace(0, 1, bins + 1)\n    result = 0.0\n    for lower, upper in zip(edges[:-1], edges[1:], strict=True):\n        selected = (confidence > lower) & (confidence <= upper)\n        if lower == 0:\n            selected |= confidence == 0\n        if selected.any():\n            result += selected.mean() * abs(\n                labels[selected].mean() - confidence[selected].mean()\n            )\n    return float(result)\n\n\n@dataclass\nclass TemperatureScaler:\n    temperature: float = 1.0\n\n    def fit(\n        self,\n        logits: np.ndarray,\n        labels: np.ndarray,\n        *,\n        steps: int = 500,\n        lr: float = 0.01,\n    ) -> float:\n        """Fit scalar temperature on validation data only using bounded gradient descent."""\n        if logits.ndim != 2 or labels.ndim != 1 or logits.shape[0] != labels.shape[0]:\n            raise ValueError("logits must be [N,C] and labels [N]")\n        log_temperature = 0.0\n        for _ in range(steps):\n            temperature = float(np.exp(log_temperature))\n            scaled = logits / temperature\n            shifted = scaled - scaled.max(axis=1, keepdims=True)\n            probabilities = np.exp(shifted)\n            probabilities /= probabilities.sum(axis=1, keepdims=True)\n            one_hot = np.eye(logits.shape[1])[labels]\n            gradient_t = np.mean(\n                np.sum((probabilities - one_hot) * (-logits / temperature**2), axis=1)\n            )\n            gradient_log_t = gradient_t * temperature\n            log_temperature -= lr * float(np.clip(gradient_log_t, -10, 10))\n            log_temperature = float(\n                np.clip(log_temperature, np.log(0.05), np.log(20.0))\n            )\n        self.temperature = float(np.exp(log_temperature))\n        return self.temperature\n\n    def transform(self, logits: np.ndarray) -> np.ndarray:\n        return logits / self.temperature\n', 'scripts/score.py': 'from __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport sys\nfrom collections import defaultdict\nfrom pathlib import Path\nfrom typing import Callable\n\nimport numpy as np\n\nREPOSITORY_ROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(REPOSITORY_ROOT / "ml"))\n\nfrom satquery_ml.evaluation import (  # noqa: E402\n    TemperatureScaler,\n    expected_calibration_error,\n)\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(\n        description=(\n            "Score raw base-vs-LoRA JSONL from the evaluation notebook without using GPU time."\n        )\n    )\n    parser.add_argument("predictions", type=Path)\n    parser.add_argument("--output", type=Path, default=Path("outputs/real-evaluation"))\n    parser.add_argument("--bootstrap-repeats", type=int, default=2000)\n    parser.add_argument("--seed", type=int, default=42)\n    return parser.parse_args()\n\n\ndef load_rows(path: Path) -> list[dict]:\n    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line]\n    if not rows:\n        raise ValueError("Prediction JSONL is empty")\n    required = {\n        "patch_id",\n        "type",\n        "base_pred",\n        "lora_pred",\n        "base_correct",\n        "lora_correct",\n        "base_sequence_confidence",\n        "lora_sequence_confidence",\n    }\n    for index, row in enumerate(rows, start=1):\n        if missing := required - set(row):\n            raise ValueError(f"Row {index} is missing fields: {sorted(missing)}")\n    return rows\n\n\ndef mean_field(rows: list[dict], field: str) -> float:\n    values = [float(row[field]) for row in rows if row.get(field) is not None]\n    return float(np.mean(values)) if values else float("nan")\n\n\ndef metric_rows(rows: list[dict], task_types: set[str]) -> list[dict]:\n    return [row for row in rows if row["type"] in task_types]\n\n\ndef bootstrap_scene_ci(\n    rows: list[dict],\n    metric: Callable[[list[dict]], float],\n    *,\n    repeats: int,\n    seed: int,\n) -> list[float]:\n    grouped: dict[str, list[dict]] = defaultdict(list)\n    for row in rows:\n        grouped[str(row["patch_id"])].append(row)\n    scenes = sorted(grouped)\n    if not scenes:\n        return [float("nan"), float("nan")]\n    rng = np.random.default_rng(seed)\n    scores = []\n    for _ in range(repeats):\n        sample = rng.choice(scenes, size=len(scenes), replace=True)\n        sample_rows = [row for scene in sample for row in grouped[str(scene)]]\n        scores.append(metric(sample_rows))\n    return [float(value) for value in np.percentile(scores, [2.5, 97.5])]\n\n\ndef summarize(rows: list[dict], prefix: str, repeats: int, seed: int) -> dict:\n    definitions = {\n        "vqa_exact_match": (\n            metric_rows(rows, {"binary", "mcq"}),\n            f"{prefix}_correct",\n        ),\n        "grounding_mean_iou": (\n            metric_rows(rows, {"bounding box"}),\n            f"{prefix}_iou",\n        ),\n        "grounding_accuracy_iou_0_5": (\n            metric_rows(rows, {"bounding box"}),\n            f"{prefix}_correct",\n        ),\n        "caption_token_f1": (\n            metric_rows(rows, {"captioning"}),\n            f"{prefix}_caption_token_f1",\n        ),\n    }\n    result = {}\n    for name, (selected, field) in definitions.items():\n        result[name] = {\n            "value": mean_field(selected, field),\n            "scene_bootstrap_95_ci": bootstrap_scene_ci(\n                selected,\n                lambda sample, field=field: mean_field(sample, field),\n                repeats=repeats,\n                seed=seed,\n            ),\n            "examples": len(selected),\n            "scenes": len({row["patch_id"] for row in selected}),\n        }\n    return result\n\n\ndef calibration_partition(row: dict) -> str:\n    digest = hashlib.sha256(str(row["patch_id"]).encode()).digest()\n    return "fit" if digest[0] < 64 else "evaluate"\n\n\ndef calibrate(rows: list[dict], prefix: str) -> dict:\n    vqa = metric_rows(rows, {"binary", "mcq"})\n    fit = [row for row in vqa if calibration_partition(row) == "fit"]\n    evaluate = [row for row in vqa if calibration_partition(row) == "evaluate"]\n    if len(fit) < 10 or len(evaluate) < 20:\n        return {\n            "status": "insufficient_data",\n            "fit_examples": len(fit),\n            "evaluation_examples": len(evaluate),\n        }\n\n    def logits_for(items: list[dict]) -> np.ndarray:\n        probability = np.clip(\n            np.asarray([row[f"{prefix}_sequence_confidence"] for row in items], dtype=float),\n            1e-6,\n            1 - 1e-6,\n        )\n        return np.stack([np.log1p(-probability), np.log(probability)], axis=1)\n\n    fit_labels = np.asarray([int(row[f"{prefix}_correct"]) for row in fit], dtype=int)\n    evaluation_labels = np.asarray(\n        [int(row[f"{prefix}_correct"]) for row in evaluate], dtype=int\n    )\n    scaler = TemperatureScaler()\n    temperature = scaler.fit(logits_for(fit), fit_labels)\n    raw_logits = logits_for(evaluate)\n    scaled_logits = scaler.transform(raw_logits)\n\n    def class_one_probability(logits: np.ndarray) -> np.ndarray:\n        shifted = logits - logits.max(axis=1, keepdims=True)\n        probability = np.exp(shifted)\n        probability /= probability.sum(axis=1, keepdims=True)\n        return probability[:, 1]\n\n    raw_probability = class_one_probability(raw_logits)\n    scaled_probability = class_one_probability(scaled_logits)\n    return {\n        "status": "within-split scene-held-out diagnostic",\n        "temperature": temperature,\n        "fit_examples": len(fit),\n        "evaluation_examples": len(evaluate),\n        "raw_ece": expected_calibration_error(raw_probability, evaluation_labels),\n        "temperature_scaled_ece": expected_calibration_error(\n            scaled_probability, evaluation_labels\n        ),\n        "meaning": (\n            "Post-hoc correctness calibration derived from real sequence scores. It becomes a "\n            "release claim only after the same frozen temperature is evaluated on an untouched "\n            "organizer-prescribed test split."\n        ),\n    }\n\n\ndef main() -> None:\n    args = parse_args()\n    if args.bootstrap_repeats < 200:\n        raise ValueError("Use at least 200 scene-bootstrap repeats")\n    rows = load_rows(args.predictions)\n    args.output.mkdir(parents=True, exist_ok=True)\n    base = summarize(rows, "base", args.bootstrap_repeats, args.seed)\n    lora = summarize(rows, "lora", args.bootstrap_repeats, args.seed)\n    deltas = {\n        name: lora[name]["value"] - base[name]["value"]\n        for name in base\n    }\n    scorecard = {\n        "source_predictions": str(args.predictions.resolve()),\n        "examples": len(rows),\n        "base": base,\n        "lora": lora,\n        "lora_minus_base": deltas,\n        "calibration": {\n            "base": calibrate(rows, "base"),\n            "lora": calibrate(rows, "lora"),\n        },\n        "scope": (\n            "Real recorded inference only. No change/fusion accuracy is reported until their "\n            "trained checkpoints and held-out labels exist."\n        ),\n    }\n    target = args.output / "scorecard.json"\n    target.write_text(json.dumps(scorecard, indent=2), encoding="utf-8")\n    print(json.dumps(scorecard, indent=2))\n    print(f"PASS: wrote {target.resolve()}")\n\n\nif __name__ == "__main__":\n    main()\n'}
for name, source in scorer_files.items():
    target = scorer_root / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding='utf-8')
scoring = runpy.run_path(str(scorer_root / 'scripts/score.py'))


In [ ]:
test_rows = scoring['load_rows'](OUTPUT_DIR / 'test_predictions.jsonl')
validation_rows = scoring['load_rows'](validation_raw)
if {r['patch_id'] for r in test_rows} & {r['patch_id'] for r in validation_rows}:
    raise ValueError('Validation/test patch leakage detected.')
validation_summary = json.loads((validation_raw.parent / 'validation_summary.json').read_text())
for key in ('adapter_revision', 'base_revision', 'dataset_revision', 'image_dataset_revision'):
    if validation_summary[key] != summary[key]:
        raise ValueError('Validation/test revision mismatch: ' + key)
test_scorecard = {p: scoring['summarize'](test_rows, p, 2000, 42) for p in ('base', 'lora')}
frozen_results = {}
for prefix in ('base', 'lora'):
    fitted = frozen_scorecard['calibration'][prefix]
    if 'temperature' not in fitted:
        frozen_results[prefix] = {'status': 'insufficient validation data; no temperature fitted'}
        continue
    rows = [r for r in test_rows if r['type'] in {'binary', 'mcq'}]
    probabilities = np.clip(np.asarray([r[prefix + '_sequence_confidence'] for r in rows]), 1e-6, 1-1e-6)
    labels = np.asarray([int(r[prefix + '_correct']) for r in rows])
    temperature = float(fitted['temperature'])
    logits = np.stack([np.log1p(-probabilities), np.log(probabilities)], axis=1) / temperature
    weights = np.exp(logits - logits.max(axis=1, keepdims=True))
    calibrated = (weights / weights.sum(axis=1, keepdims=True))[:, 1]
    ece = scoring['expected_calibration_error']
    frozen_results[prefix] = {'temperature_from_validation': temperature, 'examples': len(rows),
        'raw_ece': ece(probabilities, labels), 'frozen_temperature_test_ece': ece(calibrated, labels)}
test_scorecard['calibration_test'] = frozen_results
test_scorecard['automatic_probability_release'] = False
test_scorecard['scope'] = 'Qwen binary/MCQ correctness diagnostics; excludes mask confidence and ISRO transfer.'
(OUTPUT_DIR / 'test_scorecard.json').write_text(json.dumps(test_scorecard, indent=2))
print(json.dumps(test_scorecard, indent=2))


In [ ]:
"""Embedded in the numbered Kaggle notebooks; no repository checkout required."""

import base64
import csv
import hashlib
import io
import json
import shutil
import time
import urllib.parse
import urllib.request
from pathlib import Path


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def prepare_flood_data(destination):
    """Download only official hand-labelled triplets, with GCS generation/MD5 checks."""
    root = Path(destination)
    root.mkdir(parents=True, exist_ok=True)
    if shutil.disk_usage(root).free < 5 * 1024**3:
        raise RuntimeError(
            "Keep at least 5 GiB free for the Sen1Floods11 data and checkpoints."
        )
    base = "https://storage.googleapis.com/sen1floods11/"
    origins = []

    def fetch(object_name, target):
        metadata_url = (
            "https://storage.googleapis.com/storage/v1/b/sen1floods11/o/"
            + urllib.parse.quote(object_name, safe="")
        )
        with urllib.request.urlopen(metadata_url, timeout=60) as response:
            metadata = json.load(response)
        expected = metadata["md5Hash"]

        def matches(path):
            if not path.is_file() or path.stat().st_size != int(metadata["size"]):
                return False
            return (
                base64.b64encode(hashlib.md5(path.read_bytes()).digest()).decode()
                == expected
            )

        target.parent.mkdir(parents=True, exist_ok=True)
        if not matches(target):
            temporary = target.with_suffix(target.suffix + ".partial")
            for attempt in range(3):
                try:
                    url = base + object_name + "?generation=" + metadata["generation"]
                    with (
                        urllib.request.urlopen(url, timeout=120) as response,
                        temporary.open("wb") as out,
                    ):
                        shutil.copyfileobj(response, out)
                    if not matches(temporary):
                        raise ValueError(f"Source checksum mismatch: {object_name}")
                    temporary.replace(target)
                    break
                except Exception:
                    if attempt == 2:
                        raise
                    time.sleep(2 * (attempt + 1))
        origins.append(
            {
                "object": object_name,
                "generation": metadata["generation"],
                "md5": expected,
                "sha256": file_sha256(target),
            }
        )

    chip_ids = set()
    for split in ("train", "valid", "test"):
        source = f"v1.1/splits/flood_handlabeled/flood_{split}_data.csv"
        csv_path = root / "splits" / f"flood_{split}_data.csv"
        fetch(source, csv_path)
        identifiers = []
        for row in csv.reader(io.StringIO(csv_path.read_text(encoding="utf-8"))):
            if not row:
                continue
            name = Path(row[0].strip()).name
            if not name.endswith("_S1Hand.tif"):
                raise ValueError(f"Unexpected official split entry: {row}")
            identifier = name.removesuffix("_S1Hand.tif")
            identifiers.append(identifier)
            chip_ids.add(identifier)
        (root / "splits" / f"flood_{split}_data.txt").write_text(
            "\n".join(identifiers) + "\n", encoding="utf-8"
        )
    for number, identifier in enumerate(sorted(chip_ids), 1):
        for remote, local, suffix in (
            ("S1Hand", "S1GRDHand", "S1Hand"),
            ("S2Hand", "S2L1CHand", "S2Hand"),
            ("LabelHand", "LabelHand", "LabelHand"),
        ):
            filename = f"{identifier}_{suffix}.tif"
            fetch(
                f"v1.1/data/flood_events/HandLabeled/{remote}/{filename}",
                root / "data" / local / filename,
            )
        if number % 20 == 0 or number == len(chip_ids):
            print(
                f"Verified Sen1Floods11 triplets: {number}/{len(chip_ids)}", flush=True
            )
    (root / "source_objects.json").write_text(
        json.dumps(origins, indent=2), encoding="utf-8"
    )
    return root


def find_trained_artifacts(input_root, *, allow_experimental_segmentation=False):
    """Verify weights AND the metadata that controls preprocessing/release decisions.

    Checksums establish internal consistency, not independent authorship or model accuracy.
    """
    candidates = {"segmentation": [], "change": [], "fusion": []}
    for path in Path(input_root).rglob("config.json"):
        root = path.parent
        if not (root / "model.safetensors").is_file():
            continue
        config = json.loads(path.read_text(encoding="utf-8"))
        architecture = config.get("architecture")
        if architecture == "shared_resnet18_gru_answer_mask":
            role = "change"
        elif architecture == "terramind_s1_s2_pixel_flood_segmentation":
            role = "fusion"
        elif (
            config.get("model_type") == "segformer"
            and (root / "training_manifest.json").is_file()
        ):
            role = "segmentation"
        else:
            continue
        manifest_path = root / "sha256_manifest.json"
        if not manifest_path.is_file():
            raise ValueError(f"Missing hash manifest in {root}")
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        required = ["model.safetensors", "config.json"]
        required += (["training_manifest.json", "preprocessor_config.json"]
                     if role == "segmentation" else ["release_gate.json"])
        for name in required:
            if not (root / name).is_file() or manifest.get(name) != file_sha256(root / name):
                raise ValueError(f"Checkpoint integrity failed: {root / name}")
        if role == "segmentation":
            report = json.loads(
                (root / "training_manifest.json").read_text(encoding="utf-8")
            )
            candidate = report.get("release_candidate") is True
            if not candidate and not allow_experimental_segmentation:
                raise ValueError(
                    "SegFormer validation gate failed. Review its metrics before serving."
                )
            if not candidate:
                print("EXPERIMENTAL DEMO ONLY: SegFormer failed its release gate. "
                      "Its masks must remain labelled unvalidated; no release flag is changed.")
        else:
            gate_path = root / "release_gate.json"
            gate = (
                json.loads(gate_path.read_text(encoding="utf-8"))
                if gate_path.is_file()
                else {}
            )
            if gate.get("validation_gate_passed") is not True or gate.get("test_gate_passed") is not True:
                raise ValueError(
                    f"{role} needs passing validation/test gates from this numbered pack."
                )
        candidates[role].append(root)
    for role, roots in candidates.items():
        if len(roots) != 1:
            raise ValueError(
                f"Attach exactly one passing {role} output from notebooks 02/03/04. Found {len(roots)}. "
                "Use Kaggle Add Input → Notebook Output, or attach the extracted inference zip."
            )
    return {role: roots[0] for role, roots in candidates.items()}


def export_inference_zip(source, filename):
    import zipfile

    source, target = Path(source), Path(filename)
    with zipfile.ZipFile(target, "w", compression=zipfile.ZIP_STORED) as archive:
        for path in sorted(source.rglob("*")):
            if path.is_file() and path.name != "training_state.pt":
                archive.write(path, Path(source.name) / path.relative_to(source))
    print(f"DOWNLOAD / PRESERVE: {target}", flush=True)
    return target


In [ ]:
export_inference_zip(OUTPUT_DIR, '/kaggle/working/05_qwen_test_evidence.zip')
